# STEP 1: Data Loading

## Goal
Load and preprocess session data

## Output
Structured dataframe with time features

In [2]:
import kagglehub
import json
import pandas as pd
import os
import random

random.seed(42)

In [3]:
path = kagglehub.dataset_download("otto/recsys-dataset")
print(path)

/kaggle/input/datasets/organizations/otto/recsys-dataset


In [4]:
train_path = os.path.join(path, "otto-recsys-train.jsonl")

In [5]:
rows = []
MAX_SESSIONS = 100_000

In [6]:
with open(train_path, "r") as f:
    for i, line in enumerate(f):
        if i >= MAX_SESSIONS:
            break
        session = json.loads(line)
        for e in session["events"]:
            rows.append({
                "session": session["session"],
                "aid": e["aid"],
                "timestamp": e["ts"],
                "event_type": e["type"]
            })

In [7]:
df = pd.DataFrame(rows)

In [8]:
print(df.shape)
df.head()

(5227653, 4)


,session,aid,timestamp,event_type
0,0,1517085,1659304800025,clicks
1,0,1563459,1659304904511,clicks
2,0,1309446,1659367439426,clicks
3,0,16246,1659367719997,clicks
4,0,1781822,1659367871344,clicks


In [9]:
df["event_type"].value_counts()

event_type
clicks    4770172
carts      364579
orders      92902
Name: count, dtype: int64

In [10]:
df["datetime"] = pd.to_datetime(
    df["timestamp"],
    unit="ms",
    errors="coerce"
)

In [11]:
df["hour"] = df["datetime"].dt.hour

In [12]:
df["datetime"].isna().mean() * 100

np.float64(0.0)

In [13]:
df = df.sort_values(["session", "timestamp"])

In [14]:
events_per_session = df.groupby("session").size()
events_per_session.describe()

count    100000.00000
mean         52.27653
std          76.86172
min           2.00000
25%           6.00000
50%          19.00000
75%          63.00000
max         495.00000
dtype: float64

In [15]:
import random

random.seed(42)

# create sessions
sessions = list(df["session"].unique())

# shuffle
random.shuffle(sessions)

# split
train_sessions = sessions[:80000]
test_sessions  = sessions[80000:]

# create datasets
train_df = df[df["session"].isin(train_sessions)]
test_df  = df[df["session"].isin(test_sessions)]

# sort
train_df = train_df.sort_values(["session", "timestamp"])
test_df  = test_df.sort_values(["session", "timestamp"])

train_dict = {s: g for s, g in train_df.groupby("session")}
test_dict  = {s: g for s, g in test_df.groupby("session")}

# STEP 2: Event Weighting + Recency Modeling

## Goal
Assign importance to different user actions and prioritize recent interactions

## Output
Dataframe with event_weight, recency_weight, and intent_score

In [16]:
import numpy as np

In [17]:
EVENT_WEIGHT = {
    "clicks": 1,
    "carts": 6,
    "orders": 40
}

In [18]:
#df["event_weight"] = df["event_type"].map(EVENT_WEIGHT)
train_df["event_weight"] = train_df["event_type"].map(EVENT_WEIGHT)
test_df["event_weight"]  = test_df["event_type"].map(EVENT_WEIGHT)

In [19]:
df = df.sort_values(["session", "timestamp"])
#df["rank"] = df.groupby("session").cumcount()

In [20]:
# df["recency_weight"] = df.groupby("session")["rank"].transform(
#     lambda x: np.exp((x - x.max()) / 1.5)
# )

train_df["rank"] = train_df.groupby("session").cumcount()
train_df["recency_weight"] = train_df.groupby("session")["rank"].transform(
    lambda x: np.exp((x - x.max()) / 1.5)
)

test_df["rank"] = test_df.groupby("session").cumcount()
test_df["recency_weight"] = test_df.groupby("session")["rank"].transform(
    lambda x: np.exp((x - x.max()) / 1.5)
)

In [21]:
#df["intent_score"] = df["event_weight"] * df["recency_weight"]
train_df["intent_score"] = train_df["event_weight"] * train_df["recency_weight"]
test_df["intent_score"]  = test_df["event_weight"] * test_df["recency_weight"]

In [22]:
train_df[["event_type", "event_weight", "recency_weight", "intent_score"]].head()

,event_type,event_weight,recency_weight,intent_score
0,clicks,1,2.395218e-80,2.395218e-80
1,clicks,1,4.665247e-80,4.665247e-80
2,clicks,1,9.086660e-80,9.086660e-80
3,clicks,1,1.769840e-79,1.769840e-79
4,clicks,1,3.447177e-79,3.447177e-79


# STEP 3: Co-occurrence Matrix (Item-to-Item Relationships)

## Goal
Capture relationships between items that frequently appear together in sessions

## Output
Dictionary mapping each item to its most related items with weights

In [23]:
from collections import defaultdict

In [24]:
cooccur_count = defaultdict(lambda: defaultdict(float))

In [25]:
MAX_SESSIONS_FOR_COOCCUR = 80_000

In [26]:
from collections import defaultdict
import json

cooccur_count = defaultdict(lambda: defaultdict(float))

MAX_SESSION_LEN = 30
WINDOW = 5
MAX_ITEMS = 150000  

with open(train_path, "r") as f:
    for idx, line in enumerate(f):

        if idx % 100000 == 0:
            print("Processed:", idx)

        session = json.loads(line)
        items = [e["aid"] for e in session["events"]][-MAX_SESSION_LEN:]

        for i in range(len(items)):
            for j in range(i + 1, min(i + WINDOW, len(items))):

                a, b = items[i], items[j]

                #  LIMIT MEMORY
                if a not in cooccur_count and len(cooccur_count) > MAX_ITEMS:
                    continue

                weight = 1 / (1 + (j - i))

                cooccur_count[a][b] += weight
                cooccur_count[b][a] += weight

Processed: 0
Processed: 100000
Processed: 200000
Processed: 300000
Processed: 400000
Processed: 500000
Processed: 600000
Processed: 700000
Processed: 800000
Processed: 900000
Processed: 1000000
Processed: 1100000
Processed: 1200000
Processed: 1300000
Processed: 1400000
Processed: 1500000
Processed: 1600000
Processed: 1700000
Processed: 1800000
Processed: 1900000
Processed: 2000000
Processed: 2100000
Processed: 2200000
Processed: 2300000
Processed: 2400000
Processed: 2500000
Processed: 2600000
Processed: 2700000
Processed: 2800000
Processed: 2900000
Processed: 3000000
Processed: 3100000
Processed: 3200000
Processed: 3300000
Processed: 3400000
Processed: 3500000
Processed: 3600000
Processed: 3700000
Processed: 3800000
Processed: 3900000
Processed: 4000000
Processed: 4100000
Processed: 4200000
Processed: 4300000
Processed: 4400000
Processed: 4500000
Processed: 4600000
Processed: 4700000
Processed: 4800000
Processed: 4900000
Processed: 5000000
Processed: 5100000
Processed: 5200000
Processe

In [27]:
TOP_K = 20

In [28]:
TOP_K = 20  

cooccur = {}

for item, neighbors in cooccur_count.items():
    top_neighbors = sorted(
        neighbors.items(),
        key=lambda x: x[1],
        reverse=True
    )[:TOP_K]

    cooccur[item] = dict(top_neighbors)

In [29]:
len(cooccur)

1834009

In [30]:
import pickle

with open("cooccur_small.pkl", "wb") as f:
    pickle.dump(cooccur, f)

In [32]:
import pickle, gzip

with gzip.open("cooccur_small.pkl.gz", "wb") as f:
    pickle.dump(cooccur, f)


from IPython.display import FileLink

FileLink("cooccur_small.pkl")
#FileLink("next_top.pkl")
#FileLink("popular.pkl")

/kaggle/working/cooccur_small.pkl

# STEP 4: Candidate Generation

## Goal
Generate a strong pool of possible items to recommend for each session

## Output
Function that returns candidate items using recent history, co-occurrence, and popularity

In [29]:
# popular_items = df["aid"].value_counts().head(10000).index.tolist()
#popular_items = df["aid"].value_counts().index.tolist()
#popular_items = train_df["aid"].value_counts().index.tolist()
from collections import Counter

counter = Counter()

with open(train_path, "r") as f:
    for line in f:
        session = json.loads(line)
        for e in session["events"]:
            counter[e["aid"]] += 1

popular_items = [aid for aid, _ in counter.most_common()]

In [30]:
from collections import defaultdict

next_item = defaultdict(lambda: defaultdict(int))

with open(train_path, "r") as f:
    for idx, line in enumerate(f):

        if idx % 100000 == 0:
            print("Next-item:", idx)

        session = json.loads(line)
        items = [e["aid"] for e in session["events"]]

        for i in range(len(items) - 1):
            a, b = items[i], items[i+1]

            if a not in next_item and len(next_item) > 150000:
                continue

            next_item[a][b] += 1

Next-item: 0
Next-item: 100000
Next-item: 200000
Next-item: 300000
Next-item: 400000
Next-item: 500000
Next-item: 600000
Next-item: 700000
Next-item: 800000
Next-item: 900000
Next-item: 1000000
Next-item: 1100000
Next-item: 1200000
Next-item: 1300000
Next-item: 1400000
Next-item: 1500000
Next-item: 1600000
Next-item: 1700000
Next-item: 1800000
Next-item: 1900000
Next-item: 2000000
Next-item: 2100000
Next-item: 2200000
Next-item: 2300000
Next-item: 2400000
Next-item: 2500000
Next-item: 2600000
Next-item: 2700000
Next-item: 2800000
Next-item: 2900000
Next-item: 3000000
Next-item: 3100000
Next-item: 3200000
Next-item: 3300000
Next-item: 3400000
Next-item: 3500000
Next-item: 3600000
Next-item: 3700000
Next-item: 3800000
Next-item: 3900000
Next-item: 4000000
Next-item: 4100000
Next-item: 4200000
Next-item: 4300000
Next-item: 4400000
Next-item: 4500000
Next-item: 4600000
Next-item: 4700000
Next-item: 4800000
Next-item: 4900000
Next-item: 5000000
Next-item: 5100000
Next-item: 5200000
Next-ite

In [31]:
next_top = {}

for item, neighbors in next_item.items():
    next_top[item] = [
        n for n, _ in sorted(neighbors.items(), key=lambda x: x[1], reverse=True)[:100]
    ]

In [32]:
def generate_candidates(session_df, max_candidates=30000):

    import random

    session_items = list(session_df["aid"].values)
    recent_items = session_items[-30:]

    candidates = set(session_items)

    # 1. LAST ITEM
    last_item = session_df["aid"].iloc[-1]

    if last_item in next_top:
        candidates.update(next_top[last_item][:2000])

    if last_item in cooccur:
        candidates.update(list(cooccur[last_item].keys())[:3000])

    # 2. LAST 3 ITEMS 
    last_items = session_items[-3:]
    for item in last_items:
        if item in next_top:
            candidates.update(next_top[item][:1000])
        if item in cooccur:
            candidates.update(list(cooccur[item].keys())[:1000])

    # 3. RECENT 10 ITEMS 
    for item in recent_items[-10:]:
        if item in next_top:
            candidates.update(next_top[item][:500])
        if item in cooccur:
            candidates.update(list(cooccur[item].keys())[:500])

    # 4. LONG SESSION MEMORY 
    for item in recent_items:
        if item in cooccur:
            candidates.update(list(cooccur[item].keys())[:200])

    # 5. POPULAR BACKBONE 
    candidates.update(popular_items[:30000])

    # 6. RANDOM EXPLORATION 
    if len(candidates) < max_candidates:
        extra = random.sample(popular_items[:50000], 5000)
        candidates.update(extra)

    # 7. FINAL TRIM
    return list(candidates)[:max_candidates]

In [33]:
sample_session = test_df["session"].iloc[0]
session_df = test_df[test_df["session"] == sample_session]

In [34]:
candidates = generate_candidates(session_df)
len(candidates)

30000

In [35]:
candidates[:20]

[1310723,
 1179651,
 3,
 1048579,
 524295,
 655369,
 1572874,
 1703954,
 786469,
 38,
 1048615,
 1179688,
 1048617,
 524344,
 1310776,
 1048633,
 1704003,
 524361,
 393290,
 917585]

In [36]:
def candidate_recall_check(session_dict, samples=1000):
    import random
    random.seed(42)

    sessions = random.sample(list(session_dict.keys()), samples)

    hits = 0
    total = 0

    for s in sessions:
        s_df = session_dict[s]
        if len(s_df) < 2:
            continue

        input_df, true_item = split_session(s_df)

        candidates = generate_candidates(input_df)

        hits += int(true_item in candidates)
        total += 1

    return hits / total

In [37]:
test_dict = {s: g for s, g in test_df.groupby("session")}

In [38]:
candidate_recall_check(test_dict)

NameError: name 'split_session' is not defined

# STEP 5: Bayesian Ranking Model

## Goal
Update item probabilities dynamically based on user actions in a session

## Output
Function that computes belief scores for items using Bayesian updates

In [39]:
EVENT_LIKELIHOOD = {
    "clicks": 1.2,
    "carts": 4.0,
    "orders": 25.0
}

In [40]:
def bayesian_update(prior, event_type, aid):
    likelihood = EVENT_LIKELIHOOD[event_type]

    posterior = prior.copy()

    posterior[aid] = posterior.get(aid, 1e-6) * likelihood

    total = sum(posterior.values())
    for k in posterior:
        posterior[k] /= total

    return posterior

In [41]:
sample_session = test_df["session"].iloc[0]
session_df = test_df[test_df["session"] == sample_session].sort_values("timestamp")

In [42]:
belief = {}

for _, row in session_df.iterrows():
    belief = bayesian_update(
        belief,
        row["event_type"],
        row["aid"]
    )

In [43]:
sorted(belief.items(), key=lambda x: x[1], reverse=True)[:10]

[(424964, 0.9999515212530075),
 (105393, 4.799935488541067e-06),
 (711125, 4.799906689073288e-06),
 (215311, 4.79988249766688e-06),
 (854637, 4.799853698517041e-06),
 (1491172, 3.999844483242605e-06),
 (910862, 3.99982528408124e-06),
 (1492293, 3.999806085012033e-06),
 (718983, 1.439963021344438e-06),
 (1734061, 1.4399543816098542e-06)]

# STEP 6:Recommendation Function

## Goal
Combine Bayesian scoring, candidate generation, and frequency boost to rank items

## Output
Function that returns top-K recommended items for a session

In [44]:
def recommend_for_session(session_df, k=20):

    belief = {}

    for _, row in session_df.iterrows():
        belief = bayesian_update(
            belief,
            row["event_type"],
            row["aid"]
        )

    candidates = generate_candidates(session_df)

    for item in candidates:
        if item not in belief:
            belief[item] = 1e-9

    freq = session_df["aid"].value_counts()

    # PRECOMPUTE
    recency_map = session_df.groupby("aid")["recency_weight"].sum()
    last_item = session_df["aid"].iloc[-1]

    for aid in belief:

        score = belief.get(aid, 0)
    
        # frequency
        score *= (1 + 0.5 * freq.get(aid, 0))
    
        # recency 
        rec = recency_map.get(aid, 0)
        score *= (1 + 3 * rec)
    
        # last item
        if aid == last_item:
            score *= 5
    
        # next item
        if last_item in next_top and aid in next_top[last_item]:
            score *= 4
    
        # position boost
        last_items = session_df["aid"].values[-5:]
        if aid in last_items:
            pos = list(last_items).index(aid)
            score *= (1 + (5 - pos))
    
        # cooccur 
        co_score = 0
        for base in session_df["aid"].values[-5:]:
            if base in cooccur and aid in cooccur[base]:
                co_score += cooccur[base][aid]
    
        score *= (1 + 0.02 * co_score)
    
        belief[aid] = score
    
    total = sum(belief.values())
    if total > 0:
        for k_ in belief:
            belief[k_] /= total

    return sorted(belief.items(), key=lambda x: x[1], reverse=True)[:k]

In [45]:
sample_session = test_df["session"].iloc[0]
session_df = test_df[test_df["session"] == sample_session]

In [46]:
recommend_for_session(session_df, k=10)

[(424964, np.float64(0.9994432174023411)),
 (497868, np.float64(0.0002560933594563187)),
 (1464360, np.float64(9.941249099148445e-05)),
 (207905, np.float64(7.344435312441243e-05)),
 (376932, np.float64(6.548061644890469e-05)),
 (1628317, np.float64(6.529460536350381e-06)),
 (105393, np.float64(4.9026554871611096e-06)),
 (711125, np.float64(4.811697111337444e-06)),
 (215311, np.float64(4.799367025967724e-06)),
 (854637, np.float64(4.7976728263002115e-06))]

# STEP 7: Evaluation (Recall@K)

## Goal
Evaluate how often the true next item appears in the top-K recommendations

## Output
Recall@20 score for the recommendation system

In [47]:
def split_session(df_session):
    return df_session.iloc[:-1], df_session.iloc[-1]["aid"]

In [48]:
def trim_session(df_session, last_n=20):
    if len(df_session) > last_n:
        return df_session.iloc[-last_n:]
    return df_session

In [49]:
def predict_for_session(df_session, k=20, last_n=20):

    input_df, true_aid = split_session(df_session)
    input_df = trim_session(input_df, last_n=last_n)

    # FORCE recency recompute 
    input_df = input_df.sort_values(["timestamp"])
    input_df["rank"] = input_df.groupby("session").cumcount()
    input_df["recency_weight"] = input_df.groupby("session")["rank"].transform(
        lambda x: np.exp((x - x.max()) / 1.5)
    )

    preds = recommend_for_session(input_df, k)
    pred_items = [aid for aid, _ in preds]

    return pred_items, true_aid

In [50]:
import random
random.seed(42)

In [51]:
sessions = test_df["session"].unique()
sample_sessions = random.sample(list(sessions), 1000)

In [52]:
hits = 0
total = 0

for s in sample_sessions:
    s_df = test_dict[s]

    if len(s_df) < 2:
        continue

    preds, true_item = predict_for_session(s_df, k=20)

    hits += int(true_item in preds)
    total += 1

recall = hits / total
print("Final Recall:", recall)

Final Recall: 0.485


In [53]:
import pickle

pickle.dump(cooccur, open("cooccur.pkl", "wb"))
pickle.dump(next_top, open("next_top.pkl", "wb"))
pickle.dump(popular_items, open("popular.pkl", "wb"))

In [56]:
from IPython.display import FileLink

FileLink("cooccur.pkl")
#FileLink("next_top.pkl")
#FileLink("popular.pkl")

/kaggle/working/cooccur.pkl

In [57]:
import pickle, gzip

with gzip.open("cooccur.pkl.gz", "wb") as f:
    pickle.dump(cooccur, f)

In [1]:
import kagglehub

path = kagglehub.dataset_download("lokeshparab/amazon-products-dataset")

print("Dataset path:", path)

Dataset path: /kaggle/input/datasets/lokeshparab/amazon-products-dataset


In [2]:
import os

print(os.listdir(path))

['Gaming Consoles.csv', 'Car Electronics.csv', 'Janitorial and Sanitation Supplies.csv', 'All Electronics.csv', 'All Books.csv', 'Make-up.csv', 'Travel Accessories.csv', 'Indian Language Books.csv', 'Car and Bike Care.csv', 'Sunglasses.csv', 'Bags and Luggage.csv', 'Yoga.csv', 'Sportswear.csv', 'Fiction Books.csv', 'Exam Central.csv', 'Home Storage.csv', 'Toys Gifting Store.csv', 'All English.csv', 'Amazon-Products.csv', 'Air Conditioners.csv', 'Shoes.csv', 'Casual Shoes.csv', 'Baby Products.csv', 'Sports Collectibles.csv', 'Wallets.csv', 'Musical Instruments and Professional Audio.csv', 'Gold and Diamond Jewellery.csv', 'Nursing and Feeding.csv', 'Home Furnishing.csv', 'School Textbooks.csv', 'All Hindi.csv', 'Baby Bath Skin and Grooming.csv', 'Coffee Tea and Beverages.csv', 'Headphones.csv', 'Furniture.csv', 'Shirts.csv', 'Subscribe and Save.csv', 'Fitness Accessories.csv', 'Formal Shoes.csv', 'Cycling.csv', 'Western Wear.csv', 'Bedroom Linen.csv', 'Gaming Accessories.csv', 'Amazon F

In [3]:
import pandas as pd

df = pd.read_csv(os.path.join(path, os.listdir(path)[0]))

print(df.columns)
df.head()

Index(['name', 'main_category', 'sub_category', 'image', 'link', 'ratings',
       'no_of_ratings', 'discount_price', 'actual_price'],
      dtype='object')


,name,main_category,sub_category,image,link,ratings,no_of_ratings,discount_price,actual_price


In [4]:
df = pd.read_csv(os.path.join(path, os.listdir(path)[0]))

In [5]:
import pandas as pd
import os

df = pd.read_csv(os.path.join(path, "Amazon-Products.csv"))

print(df.shape)
df.head()

(551585, 10)


,Unnamed: 0,name,main_category,sub_category,image,link,ratings,no_of_ratings,discount_price,actual_price
0,0,Lloyd 1.5 Ton 3 Star Inverter Split Ac (5 In 1...,appliances,Air Conditioners,https://m.media-amazon.com/images/I/31UISB90sY...,https://www.amazon.in/Lloyd-Inverter-Convertib...,4.2,"2,255","₹32,999","₹58,990"
1,1,LG 1.5 Ton 5 Star AI DUAL Inverter Split AC (C...,appliances,Air Conditioners,https://m.media-amazon.com/images/I/51JFb7FctD...,https://www.amazon.in/LG-Convertible-Anti-Viru...,4.2,"2,948","₹46,490","₹75,990"
2,2,LG 1 Ton 4 Star Ai Dual Inverter Split Ac (Cop...,appliances,Air Conditioners,https://m.media-amazon.com/images/I/51JFb7FctD...,https://www.amazon.in/LG-Inverter-Convertible-...,4.2,"1,206","₹34,490","₹61,990"
3,3,LG 1.5 Ton 3 Star AI DUAL Inverter Split AC (C...,appliances,Air Conditioners,https://m.media-amazon.com/images/I/51JFb7FctD...,https://www.amazon.in/LG-Convertible-Anti-Viru...,4.0,69,"₹37,990","₹68,990"
4,4,Carrier 1.5 Ton 3 Star Inverter Split AC (Copp...,appliances,Air Conditioners,https://m.media-amazon.com/images/I/41lrtqXPiW...,https://www.amazon.in/Carrier-Inverter-Split-C...,4.1,630,"₹34,490","₹67,790"


In [9]:
products = df[["name", "image"]].dropna()

# remove broken images
products = products[products["image"].str.contains("http")]

# limit size 
products = products.head(5000)

print(products.shape)

(5000, 2)


In [10]:
products["aid"] = df["link"].astype("category").cat.codes

In [11]:
#products["aid"] = range(len(products))

In [13]:
from IPython.display import FileLink
FileLink("products.csv")

/kaggle/working/products.csv

In [12]:
products.to_csv("products.csv", index=False)

print("Saved successfully!")

Saved successfully!


# STEP 8: LightGBM Ranking Model

## Goal
Train a learning-to-rank model to improve recommendation quality using engineered features

## Output
Trained LightGBM model that ranks candidate items

In [50]:
!pip install lightgbm

In [51]:
import lightgbm as lgb
import numpy as np


In [109]:
session_dict = {s: g for s, g in df.groupby("session")}

In [53]:
!pip install tqdm

In [54]:
from tqdm import tqdm

def build_training_data(df, session_dict, num_sessions=5000):

    import random
    import numpy as np

    sessions = random.sample(list(session_dict.keys()), num_sessions)

    X, y, group = [], [], []
    global_pop = df["aid"].value_counts()

    for s in tqdm(sessions, desc="Training Data"):

        s_df = session_dict[s]

        if len(s_df) < 2:
            continue

        input_df, true_item = split_session(s_df)

        candidates = recommend_for_session(input_df, k=800)

        candidate_dict = dict(candidates)
        candidate_items = list(candidate_dict.keys())

        session_freq = input_df["aid"].value_counts()
        recency_map = input_df.groupby("aid")["recency_weight"].sum()
        session_items = input_df["aid"].unique()

        last_item = input_df["aid"].iloc[-1]

        for aid in candidate_items:

            belief_score = candidate_dict.get(aid, 0)
            freq = session_freq.get(aid, 0) / len(input_df)
            recency = recency_map.get(aid, 0)

            co_score = sum(
                cooccur.get(base, {}).get(aid, 0)
                for base in session_items
            )

            last_item_match = int(aid == last_item)

            next_item_score = int(
                last_item in next_top and aid in next_top[last_item]
            )

            features = [
                belief_score,
                freq,
                recency,
                co_score,
                last_item_match,
                next_item_score,
                global_pop.get(aid, 0)
            ]

            X.append(features)
            y.append(1 if aid == true_item else 0)

        group.append(len(candidate_items))

    return np.array(X), np.array(y), np.array(group)

In [55]:
X, y, group = build_training_data(df, session_dict, num_sessions=8000)

Training Data: 100%|██████████| 8000/8000 [36:57<00:00,  3.61it/s]  


In [56]:
np.save("X.npy", X)
np.save("y.npy", y)
np.save("group.npy", group)

In [57]:
X = np.load("X.npy")
y = np.load("y.npy")
group = np.load("group.npy")

In [58]:
params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_at": [20],
    "learning_rate": 0.008,
    "num_leaves": 127,
    "min_data_in_leaf": 50,
    "feature_fraction": 0.85,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 1.0,
    "max_depth": 10,
    "verbosity": -1
}

In [59]:
model = lgb.train(
    params,
    lgb.Dataset(X, label=y, group=group),
    num_boost_round=1000  
)

# STEP 9: Final Prediction + Evaluation (LightGBM)

## Goal
Use trained LightGBM model to rank candidates and evaluate Recall@20

## Output
Final recommendation system with improved Recall@20

In [60]:
def recommend_with_lgb(session_df, model, k=20):

    import numpy as np

    input_df, true_item = split_session(session_df)

    candidates = recommend_for_session(input_df, k=1500)
    candidate_dict = dict(candidates)
    candidate_items = list(candidate_dict.keys())

    global_pop = df["aid"].value_counts()
    session_freq = input_df["aid"].value_counts()
    recency_map = input_df.groupby("aid")["recency_weight"].sum()
    session_items = input_df["aid"].unique()

    last_item = input_df["aid"].iloc[-1]

    features = []

    for aid in candidate_items:

        belief_score = candidate_dict.get(aid, 0)
        freq = session_freq.get(aid, 0) / len(input_df)
        recency = recency_map.get(aid, 0)

        co_score = sum(
            cooccur.get(base, {}).get(aid, 0)
            for base in session_items
        )

        last_item_match = int(aid == last_item)

        next_item_score = int(
            last_item in next_top and aid in next_top[last_item]
        )

        feat = [
            belief_score,
            freq,
            recency,
            co_score,
            last_item_match,
            next_item_score,
            global_pop.get(aid, 0)
        ]

        features.append(feat)

    features = np.array(features)

    lgb_scores = model.predict(features)

    final_scores = [
        0.7 * lgb_scores[i] + 0.3 * candidate_dict.get(aid, 0)
        for i, aid in enumerate(candidate_items)
    ]

    ranked = sorted(
        zip(candidate_items, final_scores),
        key=lambda x: x[1],
        reverse=True
    )

    return [aid for aid, _ in ranked[:k]], true_item

In [61]:
import random
random.seed(42)

In [62]:
sessions = list(session_dict.keys())
sample_sessions = random.sample(sessions, 1000)

In [63]:
hits = 0
total = 0

for s in sample_sessions:
    s_df = session_dict[s]

    if len(s_df) < 2:
        continue

    preds, true_item = recommend_with_lgb(s_df, model, k=20)

    hits += int(true_item in preds)
    total += 1

recall = hits / total
recall

0.663

0.472

In [65]:
sessions = list(session_dict.keys())

train_sessions = sessions[:80000]
test_sessions  = sessions[80000:]

train_dict = {s: session_dict[s] for s in train_sessions}
test_dict  = {s: session_dict[s] for s in test_sessions}

In [66]:
X, y, group = build_training_data(df, train_dict)

Training Data: 100%|██████████| 5000/5000 [23:07<00:00,  3.60it/s]  


In [67]:
sample_sessions = random.sample(list(test_dict.keys()), 1000)

hits, total = 0, 0

for s in sample_sessions:
    s_df = test_dict[s]
    if len(s_df) < 2:
        continue

    preds, true_item = recommend_with_lgb(s_df, model)

    hits += int(true_item in preds)
    total += 1

recall = hits / total
print(recall)

0.404
